<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/06_stance_detection_binary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/gold_annotated.csv')

In [ ]:
adj_to_base = {
    1:'nicht negativ',
    2:'negativ',
    3:'nicht negativ',
}

# Apply mapping
subsample_gold['gold_standard_binary'] = subsample_gold['gold_standard'].map(adj_to_base)

In [ ]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

In [ ]:
subsample_gold.head()

##**Define Metrics**

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score
import pandas as pd

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

#**Binary Classification**

#**Binary: mlburnham/Political_DEBATE_DeBERTa_large_v1.1**

In [ ]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1')

###**Annotate Title + Paragraph**

In [ ]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']) #+ subsample_gold['lead'] + "\n\n"
template = 'Migrantinnen und Migranten werden in dem Text eher {} dargestellt.'
labels = ['als Problem', 'nicht als Problem']

In [ ]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our data frame
subsample_gold['label_mlb_p8_bin'] = [label['labels'][0] for label in res]

subsample_gold['scores_mlb_p8_bin'] = [r['scores'][0] for r in res]

In [ ]:
adj_to_base = {
    'nicht als Problem':'nicht negativ',
    'als Problem':'negativ',
}

# Apply mapping
subsample_gold['label_mlb_p8_bin'] = subsample_gold['label_mlb_p8_bin'].map(adj_to_base)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard_binary'], subsample_gold['label_mlb_p8_bin']))

In [ ]:
metrics_mlb = evaluate_model(subsample_gold['gold_standard_binary'], subsample_gold['label_mlb_p8_bin'])

print("Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1 Binary")
for metric, value in metrics_mlb.items():
    print(f"{metric}: {value:.3f}")

#**Binary: Gemma3:4b**


In [ ]:
!curl https://ollama.ai/install.sh | sh

In [ ]:
import subprocess

# Start Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Optional: check if it's running
print("Ollama server started in the background.")

In [ ]:
!curl http://localhost:11434

In [ ]:
!pip install -q ollama openai
import pandas as pd
import ollama
from ollama import chat

In [ ]:
ollama.pull("gemma3:4b")

In [ ]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json

class MigrationAnnotation(BaseModel):
   annotation: Literal['negativ', 'nicht negativ']


def classify_with_gemma_structured_binary(text):
    messages = [
    {
        "role": "system",
        "content": (
            "Stance Detection: Klassifiziere den folgenden Absatz.\n"
            "negativ: Migration wird im Text eher kritisch betrachtet.\n"
            "nicht negativ: Migration wird in dem Text positiv oder neutral betrachtet.\n"
        )
    },
    {
        "role": "user",
        "content": f"{text} --- Wie ist die Haltung des Autors zu Migration? "
    }
]

    response = chat(
        messages=messages,
        model="gemma3:4b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']

###**Annotate Title + Paragraph**

In [ ]:
# Apply the classification function the DataFrame
subsample_gold['gemma_bin'] = (subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_gemma_structured_binary)

subsample_gold['gemma_bin'] = subsample_gold['gemma_bin'].str.extract(r'"annotation"\s*:\s*"([^"]+)"')

In [ ]:
subsample_gold.head()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard_binary'], subsample_gold['gemma_bin']))

#**Evaluation Binary**

In [ ]:
metrics_gemma = evaluate_model(subsample_gold['gold_standard_binary'], subsample_gold['gemma_bin'])

print("Evaluation Metrics: Gemma3:4b Binary")
for metric, value in metrics_gemma.items():
    print(f"{metric}: {value:.3f}")

In [ ]:
metrics_df = pd.DataFrame([metrics_gemma, metrics_mlb],
                          index=['gemma3:4b_binary', 'mlburnham_binary',])

print(metrics_df.round(3))

#**Annotate Full Dataset**

In [ ]:
paragraphs_merged_df = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/df_newspaper_filtered_by_paragraph_mergedAB.csv')

In [ ]:
paragraphs_merged_df['lead'] = paragraphs_merged_df['body_text'].str.split('\n\n|\n').str[0]

In [ ]:
paragraphs_merged_df.head()

##**Load mlburnham/Political_DEBATE_DeBERTa_large_v1.1 if not already loaded**

In [ ]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1', batch_size = 8)

In [ ]:
samples = list("Title:\n" + paragraphs_merged_df['title'] + "\n\n" + paragraphs_merged_df['paragraph'])
template = 'Migrantinnen und Migranten werden in dem Text eher {} dargestellt.'
labels = ['als Problem', 'nicht als Problem']

In [ ]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to the data frame
paragraphs_merged_df['labels_NLI'] = [label['labels'][0] for label in res]

paragraphs_merged_df['scores_NLI'] = [r['scores'][0] for r in res]

In [ ]:
adj_to_base = {
    'nicht als Problem':'nicht negativ',
    'als Problem':'negativ',
}

# Apply mapping
paragraphs_merged_df['labels_NLI'] = paragraphs_merged_df['labels_NLI'].map(adj_to_base)

In [ ]:
paragraphs_merged_df['labels_NLI'].value_counts()

In [ ]:
paragraphs_merged_df.head()

##**save labels**

In [ ]:
paragraphs_merged_df.to_csv('/content/drive/MyDrive/FKM/stance_detection/df_newspaper_filtered_by_paragraph_mergedAB.csv', index=False)